# DualSight: multimodal retraining on Kaggle

Trains the DualSight model on **satellite image + numeric property features** (living area, year built, bedrooms, bathrooms, school rating, lot size), instead of the image only.

**Before you run**
1. *Settings → Accelerator →* **GPU T4 x2** (or P100). Training on CPU will take many hours.
2. *Settings →* **Internet on** (needed once to download the ImageNet weights for ResNet50; Kaggle may ask you to verify your phone number first).
3. *Add Input →* the dataset **sanveersinghsuneja/austin-satellite-property-data**.
4. *Run All*. Takes roughly 20–60 minutes on a GPU.

**Afterwards:** download **`dualsight_trained_model.zip`** from the Output panel (right side, `/kaggle/working`) and import it on your laptop with:

```
python pipeline/import_kaggle_model.py path\to\dualsight_trained_model.zip
```

Weights are exported layer-by-layer by name, so they load into TensorFlow 2.10 on your laptop whatever Keras version Kaggle uses.

In [ ]:
import os, glob, json, time, zipfile
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from PIL import Image
from sklearn.model_selection import train_test_split

layers = keras.layers
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU") or "none (this will be slow)")

## Settings

In [ ]:
QUICK_TEST = False        # True = tiny run to check the notebook works end to end
USE_LATLON = False        # True = also feed latitude/longitude (usually a big accuracy gain)

NUMERIC_FEATURES = ["livingAreaSqFt", "yearBuilt", "numOfBedrooms",
                    "numOfBathrooms", "avgSchoolRating", "lotSizeSqFt"]
if USE_LATLON:
    NUMERIC_FEATURES += ["latitude", "longitude"]

IMAGE_SIZE = 128
BATCH_SIZE = 32
INITIAL_EPOCHS = 15       # stage 1: ResNet50 frozen
FINE_TUNE_EPOCHS = 35     # stage 2: last FINE_TUNE_LAYERS of ResNet50 unfrozen
FINE_TUNE_LAYERS = 20
LR_INITIAL = 1e-3
LR_FINE_TUNE = 1e-5
SEED = 42

DATA_ROOTS = ["/kaggle/input"]
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "kaggle_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

if QUICK_TEST:
    INITIAL_EPOCHS, FINE_TUNE_EPOCHS = 1, 1
tf.keras.utils.set_random_seed(SEED)

## Find the data

In [ ]:
def find_one(pattern):
    hits = []
    for root in DATA_ROOTS:
        hits += glob.glob(os.path.join(root, "**", pattern), recursive=True)
    return sorted(hits, key=len)

csv_hits = find_one("austin_master_dataset.csv")
img_hits = find_one("property_0_rgb.png")
assert csv_hits, "austin_master_dataset.csv not found. Did you add the dataset with 'Add Input'?"
assert img_hits, "Image patches (property_*_rgb.png) not found in the attached dataset."
CSV_PATH = csv_hits[0]
IMAGE_DIR = os.path.dirname(img_hits[0])
print("CSV:   ", CSV_PATH)
print("Images:", IMAGE_DIR)

## Load, clean and split
The 80/20 split uses `random_state=42` exactly like the original project, so `pipeline/evaluate_model.py` on your laptop scores the same 20% test set.

In [ ]:
df = pd.read_csv(CSV_PATH)
df["image_file"] = df["rgb_path"].astype(str).str.replace("\\", "/", regex=False).map(os.path.basename)
available = set(os.listdir(IMAGE_DIR))
df = df[df["rgb_path"].astype(str).str.len() > 0]
df = df[df["image_file"].isin(available)].copy()
df["latestPrice"] = pd.to_numeric(df["latestPrice"], errors="coerce")
df = df.dropna(subset=["latestPrice"]).reset_index(drop=True)
for c in NUMERIC_FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["image_path"] = IMAGE_DIR + "/" + df["image_file"]
print(f"{len(df)} properties with images")

trainval_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=0.1, random_state=42)
if QUICK_TEST:
    train_idx, val_idx, test_idx = train_idx[:256], val_idx[:64], test_idx[:64]
print(f"train={len(train_idx)}  val (early stopping)={len(val_idx)}  test (untouched)={len(test_idx)}")

# Numeric features: fill gaps with the training median, standardise with training stats
medians = df.loc[train_idx, NUMERIC_FEATURES].median()
X_all = df[NUMERIC_FEATURES].fillna(medians).to_numpy("float32")
mean = X_all[train_idx].mean(axis=0)
scale = X_all[train_idx].std(axis=0)
scale[scale == 0] = 1.0
X_all = (X_all - mean) / scale
y_all = df["latestPrice"].to_numpy("float32")
paths_all = df["image_path"].to_numpy()
print(pd.DataFrame({"median": medians, "mean": mean, "std": scale}))

## Input pipeline

In [ ]:
# ResNet50 'caffe' preprocessing (same maths as keras.applications.resnet50.preprocess_input)
MEAN_BGR = tf.constant([103.939, 116.779, 123.68], dtype=tf.float32)

def load_image(path):
    img = tf.io.decode_png(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
    return tf.clip_by_value(img, 0.0, 255.0)

def caffe_preprocess(img):
    return img[..., ::-1] - MEAN_BGR

def augment(img):
    img = img / 255.0
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    img = tf.image.random_saturation(img, 0.9, 1.1)
    return tf.clip_by_value(img, 0.0, 1.0) * 255.0

def make_dataset(idx, training):
    ds = tf.data.Dataset.from_tensor_slices((paths_all[idx], X_all[idx], np.log1p(y_all[idx])))
    ds = ds.map(lambda p, t, y: (load_image(p), t, y), num_parallel_calls=tf.data.AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(idx), seed=SEED).map(lambda i, t, y: (augment(i), t, y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda i, t, y: ({"image_input": caffe_preprocess(i), "tabular_input": t}, y))
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_idx, training=True)
val_ds = make_dataset(val_idx, training=False)
test_ds = make_dataset(test_idx, training=False)

## Model
Same architecture as `pipeline/model_def.py` in the project (keep the two identical).

In [ ]:
def build_multimodal_model(num_tab_features=0, image_size=128, pretrained=True):
    image_input = layers.Input(shape=(image_size, image_size, 3), name="image_input")
    resnet = keras.applications.ResNet50(include_top=False, weights="imagenet" if pretrained else None,
                                         input_shape=(image_size, image_size, 3))
    resnet.trainable = False
    x_img = resnet(image_input, training=False)
    x_img = layers.Flatten(name="img_flatten")(x_img)
    x_img = layers.Dense(256, activation="relu", name="img_dense")(x_img)
    x_img = layers.Dropout(0.3, name="img_dropout")(x_img)
    x_img = layers.BatchNormalization(name="img_bn")(x_img)
    if num_tab_features > 0:
        tabular_input = layers.Input(shape=(num_tab_features,), name="tabular_input")
        x_num = layers.Dense(128, activation="relu", name="tab_dense")(tabular_input)
        x_num = layers.Dropout(0.2, name="tab_dropout")(x_num)
        x_num = layers.BatchNormalization(name="tab_bn")(x_num)
        fused = layers.Concatenate(name="fusion")([x_img, x_num])
        inputs = [image_input, tabular_input]
    else:
        fused = x_img
        inputs = image_input
    fused = layers.Dense(128, activation="relu", name="fused_dense")(fused)
    fused = layers.Dropout(0.3, name="fused_dropout")(fused)
    output = layers.Dense(1, activation="linear", name="price")(fused)
    model = keras.Model(inputs=inputs, outputs=output, name="satellite_price_model")
    return model

model = build_multimodal_model(len(NUMERIC_FEATURES), IMAGE_SIZE)
# Start the output at the average log-price (about 13) instead of 0, so training does not waste epochs getting there
model.get_layer("price").bias.assign([float(np.log1p(y_all[train_idx]).mean())])
backbone = next(l for l in model.layers if isinstance(l, keras.Model))
model.summary()

## Train
Stage 1 trains the new layers with ResNet50 frozen. Stage 2 fine-tunes the last ResNet50 layers at a low learning rate.

In [ ]:
t0 = time.time()
model.compile(optimizer=keras.optimizers.Adam(LR_INITIAL), loss="mse", metrics=["mae"])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=INITIAL_EPOCHS, verbose=2,
               callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)])

backbone.trainable = True
for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(LR_FINE_TUNE), loss="mse", metrics=["mae"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, verbose=2,
               callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)])
print(f"Training took {(time.time() - t0) / 60:.1f} min")

history = pd.concat([pd.DataFrame(h1.history).assign(stage=1), pd.DataFrame(h2.history).assign(stage=2)], ignore_index=True)
history.to_csv(os.path.join(OUT_DIR, "training_history.csv"), index_label="epoch")
history[["loss", "val_loss"]].plot(title="log-price MSE", figsize=(7, 3));

## Evaluate on the untouched 20% test set
`price_scale` is a smearing correction measured on the validation set. Training on log(price) makes raw predictions systematically low in dollars, and this factor corrects that.

In [ ]:
def predict_prices(ds):
    return np.expm1(model.predict(ds, verbose=0).reshape(-1).astype("float64"))

val_pred = predict_prices(val_ds)
price_scale = float(np.mean(np.exp(np.log1p(y_all[val_idx]) - np.log1p(val_pred))))

def report(actual, pred, label):
    err = pred - actual
    pct = np.abs(err) / actual * 100
    la, lp = np.log1p(actual), np.log1p(pred)
    r2 = 1 - np.sum((la - lp) ** 2) / np.sum((la - la.mean()) ** 2)
    return {"model": label, "MAE $": round(np.abs(err).mean()), "median % error": round(np.median(pct), 1),
            "within 20 %": round((pct <= 20).mean() * 100, 1), "R2 (log)": round(r2, 3),
            "bias $": round(err.mean()), "max pred $": round(pred.max())}

y_test = y_all[test_idx].astype("float64")
test_pred = predict_prices(test_ds)
rows = [report(y_test, test_pred, "multimodal (raw)"),
        report(y_test, test_pred * price_scale, "multimodal (+price_scale)"),
        {"model": "guess the median", "MAE $": round(np.abs(y_test - np.median(y_all[train_idx])).mean())}]

# Reference: numeric features only, no image (gradient boosting)
from sklearn.ensemble import HistGradientBoostingRegressor
gb = HistGradientBoostingRegressor(random_state=0).fit(X_all[train_idx], np.log1p(y_all[train_idx]))
rows.append(report(y_test, np.expm1(gb.predict(X_all[test_idx])), "numeric only (boosting)"))
results = pd.DataFrame(rows)
print(f"price_scale = {price_scale:.4f}")
results

## Export for your laptop

In [ ]:
def leaf_layers(m):
    for l in m.layers:
        if isinstance(l, keras.Model):
            yield from leaf_layers(l)
        else:
            yield l

weights = {}
for l in leaf_layers(model):
    for i, w in enumerate(l.get_weights()):
        weights[f"{l.name}::{i}"] = w
np.savez(os.path.join(OUT_DIR, "weights_by_layer.npz"), **weights)

# Reference predictions computed the same way the Flask backend does (PIL resize),
# so the import script can prove the weights arrived intact.
def pil_image(path):
    img = Image.open(path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    arr = np.asarray(img, dtype="float32")
    return arr[..., ::-1] - np.array([103.939, 116.779, 123.68], dtype="float32")

ref_idx = test_idx[:16]
ref_img = np.stack([pil_image(p) for p in paths_all[ref_idx]])
ref_log = model.predict({"image_input": ref_img, "tabular_input": X_all[ref_idx]}, verbose=0).reshape(-1)
pd.DataFrame({"row": ref_idx, "image_file": df["image_file"].to_numpy()[ref_idx], "pred_log": ref_log}).to_csv(
    os.path.join(OUT_DIR, "reference_predictions.csv"), index=False)

meta = {
    "model_kind": "multimodal",
    "numeric_features": NUMERIC_FEATURES,
    "feature_median": [float(medians[c]) for c in NUMERIC_FEATURES],
    "feature_mean": [float(v) for v in mean],
    "feature_scale": [float(v) for v in scale],
    "image_size": IMAGE_SIZE,
    "target": "log1p(latestPrice)",
    "price_scale": price_scale,
    "split": "train_test_split(test_size=0.2, random_state=42) on rows with images",
    "trained_with": {"tensorflow": tf.__version__, "keras": keras.__version__},
    "quick_test": QUICK_TEST,
    "test_results": results.to_dict(orient="records"),
}
with open(os.path.join(OUT_DIR, "model_meta.json"), "w") as f:
    json.dump(meta, f, indent=2, default=float)

zip_path = os.path.join(OUT_DIR, "dualsight_trained_model.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["weights_by_layer.npz", "model_meta.json", "reference_predictions.csv", "training_history.csv"]:
        z.write(os.path.join(OUT_DIR, name), name)
print(f"Done. Download {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB) from the Output panel.")